# Fine-tune Qwen 3.5 0.8B into a System 1 decision model

A chat model **thinks and writes** its answer word by word (System 2). A decision model such as **Jev** (TypeSafe) or
**Clef** (Cloudflare) reads the question once and returns a **typed answer with probabilities** (System 1).
We build a small one, **mini-clef**, on Qwen 3.5 0.8B: the same prompt layout as Clef, a small joint schema head and LoRA.
The task: send a consumer complaint to the right one of 9 financial products.

Inspired by Cloudflare's Clef; not made by or affiliated with Cloudflare.

In [ ]:
import json
import math
import random
import time

import torch
import torch.nn.functional as F
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm
from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration

from complaints import ROUTES, chat_messages, parse, request
from miniclef import HEAD_CONFIG, JointSchemaHead, MiniClef, collate, encode_record, save_release, systemone

## 0. Check the machine

Same **MacBook Pro** as lessons 1 and 2. PyTorch reaches the Mac's GPU through **MPS**: no NVIDIA GPU, no CUDA.

In [ ]:
!system_profiler SPHardwareDataType | grep -E "Model Name|Chip|Memory"

In [ ]:
torch.backends.mps.is_available()

## 1. The data: consumer complaints, 9 products

[`kgptalkie/cfpb-complaint-routing`](https://huggingface.co/datasets/kgptalkie/cfpb-complaint-routing): short complaints
from the US CFPB complaint database (public data, CC0), each labelled with the product it is about.
The 826 test complaints are the same ones the Laya decision model was scored on.

In [ ]:
data = load_dataset("kgptalkie/cfpb-complaint-routing")
train, test = data["train"].to_list(), data["test"].to_list()
len(train), len(test)

In [ ]:
ex = next(r for r in test if r["id"] == "cfpb-4356244")
print(ex["complaint"])
ex["product"]

## 2. The request

A Jev / SystemOne request: the **state** (what we know) and typed **questions**. A `choice` question lists its
allowed options in `criteria`, each with a short description.

In [ ]:
print(json.dumps(request(ex["complaint"]), indent=2))

## 3. The base model as System 2

Qwen 3.5 0.8B, straight from Qwen. We send it the same request as a chat message and ask for JSON back.
It has to **write** the answer, token by token.

In [ ]:
MODEL = "Qwen/Qwen3.5-0.8B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = Qwen3_5ForConditionalGeneration.from_pretrained(MODEL, dtype=torch.bfloat16).to("mps")

In [ ]:
def ask(texts):
    tok.padding_side = "left"
    prompts = [tok.apply_chat_template(chat_messages(t), add_generation_prompt=True, tokenize=False,
                                       enable_thinking=False) for t in texts]
    batch = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("mps")
    with torch.no_grad():
        out = model.generate(**batch, max_new_tokens=16, do_sample=False)
    return [tok.decode(o[batch.input_ids.shape[1]:], skip_special_tokens=True) for o in out]

In [ ]:
print(ask([ex["complaint"]])[0])

Now all 826 test complaints, 16 at a time. A reply counts when its JSON names the right product.

In [ ]:
def evaluate_chat(batch=16):
    preds = []
    for k in tqdm(range(0, len(test), batch)):
        preds += [parse(r) for r in ask([e["complaint"] for e in test[k:k + batch]])]
    return sum(p == e["product"] for p, e in zip(preds, test)) / len(test), preds.count(None)

base_acc, invalid = evaluate_chat()
base_acc, invalid

## 4. Make it System 1

**The prompt.** Clef's layout: the STATE first, then the SCHEMA with every allowed option written out as text, then
an empty `<think></think>` and `JOINT SCHEMA DECISIONS:`. `encode_record()` also remembers where the question and
each option sit in the token sequence.

In [ ]:
enc = encode_record(tok, request(ex["complaint"]))
print(tok.decode(enc.input_ids))

In [ ]:
q = enc.questions[0]
print(len(enc.input_ids), "tokens")
print("question:", tok.decode(enc.input_ids[slice(*q.question_span)]))
for oid, span in zip(q.option_ids[:3], q.option_spans):
    print(f"{oid:17}", tok.decode(enc.input_ids[slice(*span)]))

**The head.** The LLM reads this prompt **once**. Its word head (248,000 words) is never used. Instead, a small
**joint schema head** reads the hidden states: it averages the question's tokens into a question vector and each
option's tokens into an option vector, lets every option look for its evidence in the complaint, decides the questions
together, and gives each allowed option one score. A softmax turns the scores into probabilities, and plain code writes
the JSON, so it is always valid.

In [ ]:
torch.manual_seed(7)
head = JointSchemaHead(**HEAD_CONFIG).to("mps")
HEAD_CONFIG, f"{sum(p.numel() for p in head.parameters()) / 1e6:.1f}M parameters"

Untrained, the head has no idea yet: the probabilities are close to a guess.

In [ ]:
clef = MiniClef(model, head).eval()
print(json.dumps(systemone(clef, tok, request(ex["complaint"])), indent=2))

## 5. Train: LoRA + the head

LoRA adds small matrices to every attention and MLP layer of Qwen and keeps its own weights frozen. The head is new, so
it trains in full. Gradient checkpointing recomputes activations in the backward pass instead of keeping them: without
it, Qwen 3.5's linear-attention layers need about 29 GB on the Mac at batch 8.

In [ ]:
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
LORA = r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj|in_proj_qkv|in_proj_z|out_proj)"
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=LORA))
clef = MiniClef(model, head)
model.print_trainable_parameters()

Half of the training requests list only **some** of the options (always including the right one), so the head
has to read the options instead of remembering 9 fixed slots.

In [ ]:
rng = random.Random(7)

def train_request(row):
    routes = ROUTES
    if rng.random() < 0.5:
        others = [k for k in ROUTES if k != row["product"]]
        keep = {row["product"], *rng.sample(others, rng.randint(1, len(others)))}
        routes = {k: v for k, v in ROUTES.items() if k in keep}
    return request(row["complaint"], routes)

print(json.dumps(train_request(train[0])["questions"]["product"]["criteria"], indent=2))

The loss is plain cross-entropy on the right option. 100 steps of 8 complaints: **800 complaints**, a quick run.

In [ ]:
STEPS, BATCH = 100, 8
opt = torch.optim.AdamW([{"params": [p for p in model.parameters() if p.requires_grad], "lr": 2e-4},
                         {"params": head.parameters(), "lr": 1e-3}], weight_decay=0.01)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / 20) * 0.5 * (1 + math.cos(math.pi * s / STEPS)))
rows = rng.sample(train, STEPS * BATCH)

torch.mps.empty_cache()
clef.train()
start, losses = time.time(), []
for step in (bar := tqdm(range(STEPS))):
    part = rows[step * BATCH:(step + 1) * BATCH]
    encs = [encode_record(tok, train_request(r)) for r in part]
    logits = clef(collate(encs, tok.pad_token_id, "mps"))
    loss = torch.stack([F.cross_entropy(lg[0].float()[None], torch.tensor([e.questions[0].option_ids.index(r["product"])], device="mps"))
                        for r, e, lg in zip(part, encs, logits)]).mean()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(clef.parameters(), 1.0)
    opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
    losses.append(loss.item())
    bar.set_postfix(loss=f"{sum(losses[-10:]) / len(losses[-10:]):.3f}")
train_secs = time.time() - start
gpu_gb = torch.mps.driver_allocated_memory() / 1e9
print(f"{train_secs / 60:.1f} min, GPU memory {gpu_gb:.1f} GB")

## 6. Try the tuned model

The same complaint. One forward pass, a full SystemOne response, and `output_tokens` is 0: nothing was generated.

In [ ]:
clef.eval()
print(json.dumps(systemone(clef, tok, request(ex["complaint"])), indent=2))

It is not a fixed 9-way classifier: give it only 3 options and it chooses among those 3.

In [ ]:
three = {k: ROUTES[k] for k in ("mortgage", "personal_loan", "vehicle_loan")}
systemone(clef, tok, request(ex["complaint"], three))["answers"]

## 7. Before vs after

The tuned model on the same 826 test complaints, 16 at a time, then all three scores side by side.

In [ ]:
@torch.no_grad()
def evaluate(batch=16):
    right = 0
    for k in tqdm(range(0, len(test), batch)):
        part = test[k:k + batch]
        encs = [encode_record(tok, request(e["complaint"])) for e in part]
        for e, enc, lg in zip(part, encs, clef(collate(encs, tok.pad_token_id, "mps"))):
            right += enc.questions[0].option_ids[lg[0].argmax().item()] == e["product"]
    return right / len(test)

start = time.time()
tuned_acc = evaluate()
tuned_acc, f"{(time.time() - start) / len(test) * 1000:.0f} ms per complaint"

In [ ]:
from scoreboard import show
show(base_acc, tuned_acc, train_secs, gpu_gb, STEPS, BATCH)

## 8. Save and share on Hugging Face

Clef's release layout: the LoRA merged into Qwen's weights, the head next to them, and `miniclef.py` to load both.

In [ ]:
save_release("mini-clef", model, head, tok)

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
repo = f"{api.whoami()['name']}/mini-clef"
api.create_repo(repo, exist_ok=True)
api.upload_folder(folder_path="mini-clef", repo_id=repo, commit_message="mini-clef: 100 steps x 8 complaints")

## 9. Download it and use it

Anyone can now load it from the Hub with one line and send it a request.

In [ ]:
from miniclef import load_release_model
mini, mini_tok = load_release_model("kgptalkie/mini-clef")

In [ ]:
complaint = "I sent $400 with Zelle to pay my landlord, but it went to the wrong person and nobody will reverse it."
print(json.dumps(systemone(mini, mini_tok, request(complaint)), indent=2))

## Next: lesson 4

Sort support tickets automatically, with Google's **Gemma 3 270M**.